# NeuralDebug Tour

This notebook walks you through a live debugging session using NeuralDebug.
No AI agent needed — just run each cell to see how NeuralDebug works under the hood.

**What we'll do:**
1. Start a debug server on a buggy Python program
2. Set breakpoints and step through code
3. Inspect variables and evaluate expressions
4. Find the bug

**Requirements:** Python 3.8+ (no other dependencies)

In [ ]:
import json, os, socket, subprocess, sys, time

# Resolve paths relative to the repo root
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'playground' else os.getcwd()
SCRIPTS = os.path.join(REPO_ROOT, 'src', 'NeuralDebug')
EXAMPLE = os.path.join(REPO_ROOT, 'examples', 'sample_buggy_grades.py')
PORT = 15679
PYTHON = sys.executable

def send(action, args=''):
    """Send a command to the debug server and return the JSON response."""
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(30)
    sock.connect(('127.0.0.1', PORT))
    sock.sendall(json.dumps({'action': action, 'args': args}).encode())
    sock.shutdown(socket.SHUT_WR)
    chunks = []
    while True:
        try:
            data = sock.recv(65536)
            if not data: break
            chunks.append(data.decode())
        except socket.timeout: break
    sock.close()
    return json.loads(''.join(chunks))

print(f'Scripts: {SCRIPTS}')
print(f'Target:  {EXAMPLE}')
print('Ready.')

## The buggy program

Let's look at what we're debugging — a grade statistics calculator with 3 planted bugs:

In [ ]:
with open(EXAMPLE) as f:
    lines = f.readlines()
for i, line in enumerate(lines, 1):
    marker = ' ◀ BUG' if 'BUG' in line and not line.strip().startswith('#') else ''
    print(f'{i:3d}  {line.rstrip()}{marker}')

## Start the debug server

NeuralDebug uses a persistent TCP server that keeps the debugger alive across commands.
This is what lets an AI agent (or you) send commands one at a time.

In [ ]:
server = subprocess.Popen(
    [PYTHON, os.path.join(SCRIPTS, 'python_debug_session.py'),
     'serve', EXAMPLE, '--port', str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

# Wait for server
for _ in range(30):
    try:
        r = send('ping')
        if r.get('status'):
            print(f'Debug server running on port {PORT}')
            break
    except (ConnectionRefusedError, OSError):
        time.sleep(0.3)
else:
    print('ERROR: Server failed to start')

## Set a breakpoint

Line 44 is the grade filter — where the first bug lives.

In [ ]:
r = send('set_breakpoint', '44')
print(json.dumps(r, indent=2))

## Start execution

The program runs until it hits our breakpoint.

In [ ]:
r = send('start')
print(f"Status: {r['status']}")
print(f"Location: line {r.get('current_location', {}).get('line')}")
print(f"Variables: {json.dumps(r.get('local_variables', {}), indent=2)}")

## Step and inspect

Let's step through the loop iterations and watch the `valid` list grow.

In [ ]:
# Step over and inspect a few times
for i in range(3):
    r = send('continue')  # hit breakpoint again on next iteration
    loc = r.get('current_location', {})
    variables = r.get('local_variables', {})
    name_var = variables.get('name', {}).get('value', '?') if isinstance(variables.get('name'), dict) else variables.get('name', '?')
    score_var = variables.get('score', {}).get('value', '?') if isinstance(variables.get('score'), dict) else variables.get('score', '?')
    print(f"Iteration {i+1}: name={name_var}, score={score_var}, line={loc.get('line')}")

## Evaluate an expression

Ask the debugger to evaluate any Python expression in the current scope.

In [ ]:
r = send('evaluate', 'len(valid)')
print(f"len(valid) = {r.get('message', r)}")

r = send('evaluate', '[n for n, s in valid]')
print(f"names so far = {r.get('message', r)}")

## Find the bug

Continue until we hit the iteration where `score == 0` (Eve's grade).
The filter uses `score >= 0` but should use `score > 0` — so Eve's zero score sneaks through.

In [ ]:
# Keep stepping until we see score=0
for _ in range(10):
    r = send('continue')
    if r.get('status') != 'paused':
        print(f'Program finished: {r.get("status")}')
        break
    variables = r.get('local_variables', {})
    score_var = variables.get('score', {})
    score_val = score_var.get('value', score_var) if isinstance(score_var, dict) else score_var
    name_var = variables.get('name', {})
    name_val = name_var.get('value', name_var) if isinstance(name_var, dict) else name_var
    print(f'name={name_val}, score={score_val}', end='')
    if str(score_val).strip() == '0':
        print('  ← BUG! This should be excluded but passes (score >= 0)')
        break
    else:
        print()

## Clean up

In [ ]:
try:
    send('quit')
except Exception:
    pass
server.terminate()
server.wait()
print('Session ended.')

## What's next?

You just ran a full debugging session using NeuralDebug's protocol.
In practice, an AI agent sends these same JSON commands — you just write
natural language and the agent translates.

- **Connect Copilot**: see `docs/tutorials/copilot-cli.md`
- **Connect Claude**: see `docs/tutorials/claude-mcp.md`
- **Try C/C++**: `python src/NeuralDebug/cpp_debug_session.py info`
- **Build your own agent**: see `integrations/README.md`